# Patching and regular updates  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 4.3.c: Patch-prioritization engine

**Skill:** Mitigate Risk · **Maps to:** 4.3.B, 4.3.C · **Language:** Python 3 (pytest)

## Problem
A patch-management team needs to triage a queue of pending vulnerabilities and decide how
urgently each must be addressed. Each vulnerability has a CVSS severity score, a flag for
whether it is being actively exploited in the wild, a flag for whether the affected asset is
internet-facing, and a flag for whether the affected software is end-of-life (no patch will ever
be released). Build the triage engine that assigns one urgency label to each vulnerability.

## Function contract
Implement `def solve(vulns: list) -> list` in the code editor (see the Reference section below).

## Input
A list of dicts, each with keys: `id` (str), `cvss` (float, 0.0-10.0), `exploited_in_wild`
(bool), `internet_facing` (bool), `end_of_life` (bool).

## Output
A list of dicts, one per input vulnerability (same order as input), each with keys `id` (str,
copied from input) and `urgency` (str), where `urgency` is one of: `"emergency"`,
`"scheduled"`, `"routine"`, `"compensate"`.

## Decision rule (apply in this exact order, first match wins, per vulnerability)
1. `end_of_life` is `True` → `"compensate"` (no patch will ever exist; patching is not an
   option regardless of severity or exploitation status).
2. Else, `exploited_in_wild` is `True` AND `internet_facing` is `True` → `"emergency"`.
3. Else, `cvss >= 9.0` AND `internet_facing` is `True` → `"emergency"` (a working exploit is
   likely imminent for a critical, internet-facing flaw even without confirmed in-the-wild
   activity).
4. Else, `cvss >= 9.0` → `"scheduled"` (critical severity but not internet-facing).
5. Else, `cvss >= 7.0` AND `internet_facing` is `True` → `"scheduled"`.
6. Else → `"routine"`.

Rule order matters: `end_of_life` is checked first and overrides every other condition, an
end-of-life system is never `"emergency"` or `"scheduled"` even at CVSS 10.0, because patching is
not a valid remediation at all.

## Examples
```
[{"id": "V1", "cvss": 9.1, "exploited_in_wild": True, "internet_facing": True, "end_of_life": False}]
    -> [{"id": "V1", "urgency": "emergency"}]

[{"id": "V2", "cvss": 8.8, "exploited_in_wild": False, "internet_facing": False, "end_of_life": True}]
    -> [{"id": "V2", "urgency": "compensate"}]

[{"id": "V3", "cvss": 4.2, "exploited_in_wild": False, "internet_facing": False, "end_of_life": False}]
    -> [{"id": "V3", "urgency": "routine"}]
```

## Constraints
- Deterministic, offline, standard library only.
- Preserve input order in the output list.
- Exactly one urgency label per vulnerability.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(vulns: list) -> list
```

### Input (`vulns`)
A list of dicts, each describing one vulnerability:

| key | type | meaning |
|---|---|---|
| `id` | str | vulnerability identifier, echoed unchanged into the output |
| `cvss` | float | CVSS severity score, 0.0-10.0 |
| `exploited_in_wild` | bool | whether the vulnerability is confirmed being actively exploited |
| `internet_facing` | bool | whether the affected asset is reachable from the public internet |
| `end_of_life` | bool | whether the affected software is end-of-life (no patch will ever ship) |

### Output (`result`)
A list of dicts, same length and same order as the input, each with:
- `"id"`: the same id string from the corresponding input entry
- `"urgency"`: one of `"emergency"`, `"scheduled"`, `"routine"`, `"compensate"`

### Decision rule (exact priority order, evaluate top to bottom per vulnerability, return on
first match)
1. If `end_of_life` is `True` → `"compensate"`. This overrides everything else, no CVSS score
   or exploitation status changes this, because patching an end-of-life product is not possible.
2. Else if `exploited_in_wild` is `True` and `internet_facing` is `True` → `"emergency"`.
3. Else if `cvss >= 9.0` and `internet_facing` is `True` → `"emergency"`.
4. Else if `cvss >= 9.0` → `"scheduled"`.
5. Else if `cvss >= 7.0` and `internet_facing` is `True` → `"scheduled"`.
6. Else → `"routine"`.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only unless the hidden grading tests: allow_deps` lists otherwise.
- CVSS thresholds are inclusive (`>=`), not strict.
- Input order is preserved in the output.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 4.3.c. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""


def solve(vulns: list) -> list:
    """Assign an urgency label to each vulnerability per the patch-triage priority rules.

    Args:
        vulns: list of dicts with keys id (str), cvss (float), exploited_in_wild (bool),
            internet_facing (bool), end_of_life (bool).

    Returns:
        list of dicts, same order, each {"id": ..., "urgency": ...} where urgency is one of
        "emergency", "scheduled", "routine", "compensate".
    """
    results = []
    for v in vulns:
        vid = v["id"]
        cvss = v.get("cvss", 0.0)
        exploited = v.get("exploited_in_wild", False)
        internet_facing = v.get("internet_facing", False)
        end_of_life = v.get("end_of_life", False)

        if end_of_life:
            urgency = "compensate"
        elif exploited and internet_facing:
            urgency = "emergency"
        elif cvss >= 9.0 and internet_facing:
            urgency = "emergency"
        elif cvss >= 9.0:
            urgency = "scheduled"
        elif cvss >= 7.0 and internet_facing:
            urgency = "scheduled"
        else:
            urgency = "routine"

        results.append({"id": vid, "urgency": urgency})

    return results


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_example_1_emergency_exploited_internet_facing():
    vulns = [{"id": "V1", "cvss": 9.1, "exploited_in_wild": True,
              "internet_facing": True, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V1", "urgency": "emergency"}]


def test_example_2_compensate_end_of_life_overrides_severity():
    vulns = [{"id": "V2", "cvss": 8.8, "exploited_in_wild": False,
              "internet_facing": False, "end_of_life": True}]
    assert impl.solve(vulns) == [{"id": "V2", "urgency": "compensate"}]


def test_example_3_routine_low_severity():
    vulns = [{"id": "V3", "cvss": 4.2, "exploited_in_wild": False,
              "internet_facing": False, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V3", "urgency": "routine"}]


# ---- hidden / edge cases -------------------------------------------------
def test_edge_empty_list():
    assert impl.solve([]) == []


def test_critical_internal_is_scheduled_not_emergency():
    vulns = [{"id": "V4", "cvss": 9.8, "exploited_in_wild": False,
              "internet_facing": False, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V4", "urgency": "scheduled"}]


def test_critical_internet_facing_not_yet_exploited_is_emergency():
    vulns = [{"id": "V5", "cvss": 9.0, "exploited_in_wild": False,
              "internet_facing": True, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V5", "urgency": "emergency"}]


def test_moderate_severity_internet_facing_is_scheduled():
    vulns = [{"id": "V6", "cvss": 7.5, "exploited_in_wild": False,
              "internet_facing": True, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V6", "urgency": "scheduled"}]


def test_moderate_severity_internal_is_routine():
    vulns = [{"id": "V7", "cvss": 7.0, "exploited_in_wild": False,
              "internet_facing": False, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V7", "urgency": "routine"}]


def test_end_of_life_overrides_exploited_and_internet_facing():
    vulns = [{"id": "V8", "cvss": 10.0, "exploited_in_wild": True,
              "internet_facing": True, "end_of_life": True}]
    assert impl.solve(vulns) == [{"id": "V8", "urgency": "compensate"}]


def test_boundary_cvss_exactly_seven_internet_facing():
    vulns = [{"id": "V9", "cvss": 7.0, "exploited_in_wild": False,
              "internet_facing": True, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V9", "urgency": "scheduled"}]


def test_boundary_cvss_just_below_seven_internet_facing_is_routine():
    vulns = [{"id": "V10", "cvss": 6.9, "exploited_in_wild": False,
              "internet_facing": True, "end_of_life": False}]
    assert impl.solve(vulns) == [{"id": "V10", "urgency": "routine"}]


def test_order_preserved_and_multiple_entries():
    vulns = [
        {"id": "A", "cvss": 2.0, "exploited_in_wild": False,
         "internet_facing": False, "end_of_life": False},
        {"id": "B", "cvss": 9.9, "exploited_in_wild": True,
         "internet_facing": True, "end_of_life": False},
        {"id": "C", "cvss": 5.0, "exploited_in_wild": False,
         "internet_facing": False, "end_of_life": True},
    ]
    assert impl.solve(vulns) == [
        {"id": "A", "urgency": "routine"},
        {"id": "B", "urgency": "emergency"},
        {"id": "C", "urgency": "compensate"},
    ]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
